# 10 — Estimação AIPW V2

**Objetivo.** Estimar, ano a ano (2014–2024, sem *pooled*), o contraste AIPW padronizado do risco de baixo peso ao nascer (`PESO < 2.500 g`) entre início do pré-natal nos meses 1–3 e um comparador restrito: **B** (4–6, V2 principal) e **C** (4–5, sensibilidade). Antes de B/C, o contraste histórico **A** (1–3 vs 4–9) de 2024 é reproduzido com nuisances reajustados do zero (controle positivo).

As estimativas são contrastes observacionais padronizados **sob hipóteses de identificação**; não constituem prova causal.

## 1. Contrato congelado

O desenho segue o [contrato V2](../../docs/temporal/CONTRATO_V2_PRE_ESTIMACAO.md) (hash científico `2174989c…`) e o contrato AIPW do [notebook 09](09_estabilidade_temporal_estimativas.ipynb); nada foi redefinido aqui.

| Item | Contrato |
|---|---|
| População | nascidos vivos elegíveis da V1 (gestação única, peso 500–6.000 g, `MESPRENAT` válido); B mantém meses 1–6, C mantém 1–5 |
| `MESPRENAT` 99, vazio, 10, inválidos | fora de T, sem reclassificação |
| X | sete dimensões congeladas (idade, escolaridade, raça/cor, situação conjugal, paridade, perdas fetais, UF) |
| Ajuste cruzado | 3 partições estratificadas por `2*T+Y`, seed `20240925`; pré-processamento ajustado só no treino |
| Ordem | 2024: `ORDER BY contador`; 2014–2023: `UF_RESIDENCIA, CAST(CONTADOR AS VARCHAR)`; B/C são subconjuntos que preservam essa ordem |
| C1 / C2 / C3 | logística L2 `C=1` + desfechos logísticos / mesma logística + HGB histórico / HGB histórico + HGB histórico |
| Recorte de suporte | principal sem recorte; sensibilidade `[0,05; 0,95]`; `[0,01; 0,99]` só como contagem |
| Inferência | principal: SE agrupado por `CODMUNRES`, IC com t(G−1); referência: iid |

Os nuisances de B e C são reajustados em cada população, ano e partição. Nenhuma predição da V1 ou do diagnóstico 09d é reutilizada; o cache próprio fica em `data/interim/estimacao_v2/`.

In [ ]:

from pathlib import Path
import sys
import json

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display


# ============================================================
# Localiza a raiz do repositório
# ============================================================

cwd = Path.cwd().resolve()

raiz_repo = None

for caminho in [cwd, *cwd.parents]:
    if (caminho / "src" / "estimacao_v2.py").exists():
        raiz_repo = caminho
        break

if raiz_repo is None:
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto. "
        f"Diretório atual do kernel: {cwd}"
    )


# ============================================================
# Adiciona a raiz ao PYTHONPATH da sessão
# ============================================================

if str(raiz_repo) not in sys.path:
    sys.path.insert(0, str(raiz_repo))


# ============================================================
# Kernel: os checkpoints exigem as versões da .venv do projeto
# ============================================================

venv_projeto = (raiz_repo / ".venv").resolve()
print("Python do kernel:", sys.executable)
if Path(sys.prefix).resolve() != venv_projeto:
    print(
        f"ATENÇÃO: kernel fora da .venv do projeto ({venv_projeto}).\n"
        "Selecione .venv\\Scripts\\python.exe como kernel; com outras versões de "
        "Python/scikit-learn/NumPy/DuckDB os checkpoints são rejeitados."
    )


# ============================================================
# Imports do projeto
# ============================================================

from src.estimacao_v2 import RAIZ, PASTA_SAIDA, executar
from src.visualizacao import aplicar_tema_ipt


# ============================================================
# Configuração
# ============================================================

raiz = RAIZ
pasta = raiz / PASTA_SAIDA

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)


print("Diretório atual do kernel:", cwd)
print("Raiz localizada:", raiz_repo)
print("Raiz definida pelo projeto:", raiz)
print("Pasta de saída:", pasta)

Diretório atual do kernel: C:\GitHub\data-science-projects\projetos\big-data-sinasc-causal-ml\notebooks
Raiz localizada: C:\GitHub\data-science-projects\projetos\big-data-sinasc-causal-ml
Raiz definida pelo projeto: C:\GitHub\data-science-projects\projetos\big-data-sinasc-causal-ml
Pasta de saída: C:\GitHub\data-science-projects\projetos\big-data-sinasc-causal-ml\outputs\temporal\estimacao_v2


## 2. Execução

`executar` reproduz primeiro A/2024 e só segue para B/C se a reconciliação passar. Cada ajuste é lido do checkpoint quando o contrato completo (formulação, ano, partição, modelo, seed, X, ordem, hashes de dados/T/Y, índices, pré-processamento, versões e código) é idêntico; caso contrário o checkpoint é rejeitado.

In [2]:
resultado = executar(raiz)
manifesto = resultado['manifesto']
controle = resultado['controle']
print('Controle positivo:', controle['status'])
print('Ajustes novos nesta sessão:', manifesto['cache']['ajustes_novos'],
      '| reutilizados:', manifesto['cache']['ajustes_reutilizados'])
print('Tempo de ajuste registrado nos checkpoints (min):', round(manifesto['cache']['segundos_fit_checkpoints'] / 60, 1))
assert controle['status'] == 'CONTROLE_POSITIVO_A2024_APROVADO', controle['falhas']

2024 | preparo | N=2251570 | 22.0s | RSS=1627529216


ValueError: Checkpoint incompatível: C:\GitHub\data-science-projects\projetos\big-data-sinasc-causal-ml\data\interim\estimacao_v2\codigo_bb3d74ec91827625\A\2024\e_fold1

## 3. Controle positivo A/2024

Os 18 nuisances de A/2024 foram reajustados do zero e comparados, na tolerância histórica `1e-12`, com: N/T0/T1 esperados; hashes de Parquet, chave, T e Y do notebook 09; tamanhos e hashes dos índices das três partições da Fase 2; OOF da Fase 2/3 (apenas lido); C1/C2/C3 com e sem recorte; SE iid, SE municipal, IC e AUC dos escores.

In [ ]:
grupos = {'População e hashes': ('esperado_', 'notebook09_chave', 'notebook09_t_', 'notebook09_y_', 'notebook09_parquet', 'notebook09_n_clusters', 'n', 'parquet', 'features'),
          'Partições': ('fold',), 'OOF refeito vs histórico': ('oof_',), 'AUC dos escores': ('C1_auc', 'C3_auc')}
linhas = []
for nome, prefixos in grupos.items():
    chaves = [k for k in controle['checks'] if k.startswith(prefixos)]
    linhas.append({'grupo': nome, 'checks': len(chaves), 'aprovados': sum(controle['checks'][k] for k in chaves)})
linhas.append({'grupo': 'Total (inclui efeitos, SE e IC)', 'checks': controle['n_checks'], 'aprovados': controle['n_checks'] - len(controle['falhas'])})
display(pd.DataFrame(linhas))
oof = {k.replace('oof_', '').replace('_max_abs', ''): v for k, v in controle['deltas'].items() if k.startswith('oof_')}
print('Maior |diferença| no OOF refeito por nuisance:', oof)
print('Maior |delta| em qualquer comparação:', controle['max_abs_delta'])
res = pd.read_csv(pasta / 'resultados_anuais.csv')
display(res.loc[res.formulacao.eq('A'), ['especificacao', 'trimming', 'n', 'n_excluido', 'estimativa_pp', 'se_iid_pp', 'se_cluster_pp',
                                         'ic95_cluster_inferior_pp', 'ic95_cluster_superior_pp']].round(6))

,grupo,checks,aprovados
0,População e hashes,47,47
1,Partições,13,13
2,OOF refeito vs histórico,6,6
3,AUC dos escores,2,2
4,"Total (inclui efeitos, SE e IC)",141,141


Maior |diferença| no OOF refeito por nuisance: {'e': 0.0, 'm0_C1': 0.0, 'm1_C1': 0.0, 'm0_C2': 0.0, 'm1_C2': 0.0, 'e_C3': 0.0}
Maior |delta| em qualquer comparação: 2.220446049250313e-16


,especificacao,trimming,n,n_excluido,estimativa_pp,se_iid_pp,se_cluster_pp,ic95_cluster_inferior_pp,ic95_cluster_superior_pp
0,C1,sem_trimming,2251570,0,-1.340527,0.061994,0.068287,-1.474397,-1.206657
1,C1,0.05_0.95,2103319,148251,-1.382662,0.060840,0.068626,-1.517196,-1.248127
2,C2,sem_trimming,2251570,0,-1.231456,0.061981,0.067997,-1.364757,-1.098155
3,C2,0.05_0.95,2103319,148251,-1.273369,0.060813,0.068371,-1.407403,-1.139335
4,C3,sem_trimming,2251570,0,-1.232469,0.061044,0.067628,-1.365046,-1.099892
5,C3,0.05_0.95,2096706,154864,-1.250146,0.060684,0.069204,-1.385813,-1.114480


## 4. Populações B e C

N, braços e municípios por ano. O status de suporte vem do 09d (congelado): B 2024 e C 2017–2024 estão **sinalizados** por caudas do escore acima de 10% fora de `[0,05; 0,95]`; nenhum ano está bloqueado, e todos foram estimados.

In [ ]:
execucao = pd.read_csv(pasta / 'execucao.csv')
pop = execucao.loc[execucao.formulacao.isin(['B', 'C'])].pivot(index='ano', columns='formulacao', values=['n', 'n_t0', 'n_t1', 'n_municipios', 'status_suporte_09d'])
pop.columns = [f'{f}_{c}' for c, f in pop.columns]
display(pop[[f'{f}_{c}' for f in ('B', 'C') for c in ('n', 'n_t0', 'n_t1', 'n_municipios', 'status_suporte_09d')]])

,B_n,B_n_t0,B_n_t1,B_n_municipios,B_status_suporte_09d,C_n,C_n_t0,C_n_t1,C_n_municipios,C_status_suporte_09d
ano,,,,,,,,,,
2014,2615773,501781,2113992,5583,APROVADO,2538604,424612,2113992,5583,APROVADO
2015,2673272,493021,2180251,5587,APROVADO,2599919,419668,2180251,5587,APROVADO
2016,2560827,452543,2108284,5586,APROVADO,2492767,384483,2108284,5586,APROVADO
2017,2630828,446919,2183909,5581,APROVADO,2562927,379018,2183909,5581,SINALIZADO
2018,2672423,437401,2235022,5582,APROVADO,2606245,371223,2235022,5582,SINALIZADO
2019,2586743,406953,2179790,5580,APROVADO,2524898,345108,2179790,5580,SINALIZADO
2020,2476058,385034,2091024,5583,APROVADO,2415861,324837,2091024,5583,SINALIZADO
2021,2425844,343749,2082095,5586,APROVADO,2371495,289400,2082095,5586,SINALIZADO
2022,2344255,325836,2018419,5586,APROVADO,2291516,273097,2018419,5586,SINALIZADO


## 5. Resultados principais

Contraste AIPW em pontos percentuais (p.p.), sem recorte de suporte, com IC95% municipal. Valores negativos indicam menor risco padronizado de baixo peso no início em 1–3.

In [ ]:
sem = res.loc[res.trimming.eq('sem_trimming')].copy()
sem['ic_municipal'] = sem.apply(lambda r: f"[{r.ic95_cluster_inferior_pp:.3f}; {r.ic95_cluster_superior_pp:.3f}]", axis=1)
for f in ('B', 'C'):
    tab = sem.loc[sem.formulacao.eq(f)].pivot(index='ano', columns='especificacao', values=['estimativa_pp', 'ic_municipal'])
    tab.columns = [f'{e} {c}' for c, e in tab.columns]
    tab = tab[[f'{e} {c}' for e in ('C1', 'C2', 'C3') for c in ('estimativa_pp', 'ic_municipal')]]
    tab.insert(0, 'suporte 09d', sem.loc[sem.formulacao.eq(f) & sem.especificacao.eq('C1')].set_index('ano').status_suporte_09d)
    print(f'Formulação {f}')
    display(tab.round(3))

Formulação B

,suporte 09d,C1 estimativa_pp,C1 ic_municipal,C2 estimativa_pp,C2 ic_municipal,C3 estimativa_pp,C3 ic_municipal
ano,,,,,,,
2014,APROVADO,-0.936356,[-1.046; -0.827],-0.859483,[-0.968; -0.751],-0.852809,[-0.962; -0.743]
2015,APROVADO,-1.003118,[-1.111; -0.896],-0.912568,[-1.020; -0.805],-0.919982,[-1.025; -0.815]
2016,APROVADO,-1.089764,[-1.230; -0.949],-0.994438,[-1.137; -0.852],-0.990056,[-1.130; -0.850]
2017,APROVADO,-1.125742,[-1.268; -0.984],-1.043766,[-1.187; -0.900],-1.036321,[-1.178; -0.895]
2018,APROVADO,-1.256143,[-1.409; -1.103],-1.176714,[-1.330; -1.023],-1.169095,[-1.321; -1.018]
2019,APROVADO,-1.210294,[-1.339; -1.082],-1.122368,[-1.250; -0.995],-1.109147,[-1.233; -0.986]
2020,APROVADO,-1.065976,[-1.210; -0.922],-0.992083,[-1.136; -0.848],-0.989069,[-1.134; -0.845]
2021,APROVADO,-1.251769,[-1.394; -1.110],-1.170548,[-1.310; -1.031],-1.165638,[-1.302; -1.030]
2022,APROVADO,-1.523921,[-1.671; -1.377],-1.433433,[-1.580; -1.287],-1.432475,[-1.577; -1.288]


Formulação C


,suporte 09d,C1 estimativa_pp,C1 ic_municipal,C2 estimativa_pp,C2 ic_municipal,C3 estimativa_pp,C3 ic_municipal
ano,,,,,,,
2014,APROVADO,-0.851492,[-0.963; -0.740],-0.772896,[-0.885; -0.661],-0.770457,[-0.882; -0.659]
2015,APROVADO,-0.952955,[-1.066; -0.840],-0.863775,[-0.977; -0.750],-0.872188,[-0.984; -0.760]
2016,APROVADO,-1.017514,[-1.154; -0.881],-0.930938,[-1.069; -0.793],-0.925887,[-1.063; -0.789]
2017,SINALIZADO,-1.027532,[-1.171; -0.884],-0.952532,[-1.096; -0.809],-0.940771,[-1.085; -0.796]
2018,SINALIZADO,-1.178571,[-1.333; -1.025],-1.110099,[-1.264; -0.956],-1.10386,[-1.256; -0.952]
2019,SINALIZADO,-1.159512,[-1.298; -1.021],-1.077442,[-1.216; -0.939],-1.067101,[-1.201; -0.933]
2020,SINALIZADO,-0.962831,[-1.112; -0.814],-0.891912,[-1.042; -0.742],-0.887957,[-1.037; -0.739]
2021,SINALIZADO,-1.1827,[-1.327; -1.038],-1.104176,[-1.247; -0.961],-1.100521,[-1.240; -0.961]
2022,SINALIZADO,-1.487046,[-1.633; -1.341],-1.410652,[-1.556; -1.265],-1.410146,[-1.552; -1.268]


In [ ]:
cores = {'C1': '#00598E', 'C2': '#04B4E3', 'C3': '#9B3C8B'}

def figura_formulacao(f, titulo):
    fig = go.Figure()
    for i, spec in enumerate(('C1', 'C2', 'C3')):
        tab = sem.loc[sem.formulacao.eq(f) & sem.especificacao.eq(spec)].sort_values('ano')
        fig.add_trace(go.Scatter(x=tab.ano + (i - 1) * 0.12, y=tab.estimativa_pp, mode='lines+markers', name=spec,
            line=dict(color=cores[spec]), marker=dict(size=8, symbol=['diamond' if s else 'circle' for s in tab.sinalizado_09d]),
            error_y=dict(type='data', symmetric=False, array=tab.ic95_cluster_superior_pp - tab.estimativa_pp,
                         arrayminus=tab.estimativa_pp - tab.ic95_cluster_inferior_pp),
            customdata=tab[['ano', 'n', 'status_suporte_09d']].to_numpy(),
            hovertemplate='Ano %{customdata[0]}<br>%{y:.3f} p.p.<br>N %{customdata[1]:,}<br>Suporte %{customdata[2]}<extra>' + spec + '</extra>'))
    fig.add_hline(y=0, line_color='#888', line_width=1)
    fig.update_layout(xaxis_title='Ano', yaxis_title='Contraste AIPW (p.p.), IC95% municipal', height=450)
    fig.update_xaxes(dtick=1)
    return aplicar_tema_ipt(fig, titulo)

figura_formulacao('B', 'B (1–3 vs 4–6): contraste anual').show()

In [ ]:
figura_formulacao('C', 'C (1–3 vs 4–5): contraste anual').show()

Comparação B × C em C1, com a V1 histórica (A, notebook 09) apenas como referência visual. Losangos marcam anos sinalizados no 09d. As três séries têm populações e comparadores distintos; a sobreposição dos ICs não é um teste de diferença.

In [ ]:
v1 = pd.read_csv(raiz / 'outputs/temporal/estimativas/aipw_anual.csv')
v1 = v1.loc[v1.especificacao.eq('C1') & v1.trimming.eq('sem_trimming')].sort_values('ano')
fig = go.Figure()
fig.add_trace(go.Scatter(x=v1.ano, y=v1.estimativa_pp, mode='lines', name='A (V1, notebook 09)',
                         line=dict(color='#9A9A9A', dash='dot'), hovertemplate='%{y:.3f} p.p.<extra>A</extra>'))
for i, (f, cor) in enumerate((('B', '#00598E'), ('C', '#E07A1F'))):
    tab = sem.loc[sem.formulacao.eq(f) & sem.especificacao.eq('C1')].sort_values('ano')
    fig.add_trace(go.Scatter(x=tab.ano + (i - .5) * 0.15, y=tab.estimativa_pp, mode='lines+markers', name=f,
        line=dict(color=cor), marker=dict(size=8, symbol=['diamond' if s else 'circle' for s in tab.sinalizado_09d]),
        error_y=dict(type='data', symmetric=False, array=tab.ic95_cluster_superior_pp - tab.estimativa_pp,
                     arrayminus=tab.estimativa_pp - tab.ic95_cluster_inferior_pp),
        hovertemplate='%{y:.3f} p.p.<extra>' + f + '</extra>'))
fig.add_hline(y=0, line_color='#888', line_width=1)
fig.update_layout(xaxis_title='Ano', yaxis_title='Contraste AIPW C1 (p.p.)', height=450)
fig.update_xaxes(dtick=1)
aplicar_tema_ipt(fig, 'B × C ao longo do tempo (C1)').show()

## 6. Sensibilidade de modelo (C1, C2, C3)

Amplitude entre as três especificações por ano (máximo − mínimo, em p.p.), sem recorte de suporte.

In [ ]:
amp = sem.pivot_table(index=['formulacao', 'ano'], columns='especificacao', values='estimativa_pp')
amp['amplitude_pp'] = amp[['C1', 'C2', 'C3']].max(axis=1) - amp[['C1', 'C2', 'C3']].min(axis=1)
amp['C2_menos_C1'] = amp.C2 - amp.C1
amp['C3_menos_C2'] = amp.C3 - amp.C2
display(amp.loc[['B', 'C']].round(3))
print(amp.loc[['B', 'C']].groupby(level=0)[['amplitude_pp', 'C2_menos_C1', 'C3_menos_C2']].agg(['min', 'max']).round(3))

especificacao       C1     C2     C3  amplitude_pp  C2_menos_C1  C3_menos_C2
formulacao ano                                                              
B          2014 -0.936 -0.859 -0.853         0.084        0.077        0.007
           2015 -1.003 -0.913 -0.920         0.091        0.091       -0.007
           2016 -1.090 -0.994 -0.990         0.100        0.095        0.004
           2017 -1.126 -1.044 -1.036         0.089        0.082        0.007
           2018 -1.256 -1.177 -1.169         0.087        0.079        0.008
           2019 -1.210 -1.122 -1.109         0.101        0.088        0.013
           2020 -1.066 -0.992 -0.989         0.077        0.074        0.003
           2021 -1.252 -1.171 -1.166         0.086        0.081        0.005
           2022 -1.524 -1.433 -1.432         0.091        0.090        0.001
           2023 -1.429 -1.339 -1.332         0.097        0.090        0.007
           2024 -1.480 -1.373 -1.361         0.120        0.108        0.012
C          2014 -0.851 -0.773 -0.770         0.081        0.079        0.002
           2015 -0.953 -0.864 -0.872         0.089        0.089       -0.008
           2016 -1.018 -0.931 -0.926         0.092        0.087        0.005
           2017 -1.028 -0.953 -0.941         0.087        0.075        0.012
           2018 -1.179 -1.110 -1.104         0.075        0.068        0.006
           2019 -1.160 -1.077 -1.067         0.092        0.082        0.010
           2020 -0.963 -0.892 -0.888         0.075        0.071        0.004
           2021 -1.183 -1.104 -1.101         0.082        0.079        0.004
           2022 -1.487 -1.411 -1.410         0.077        0.076        0.001
           2023 -1.333 -1.240 -1.236         0.097        0.093        0.004
           2024 -1.330 -1.215 -1.217         0.115        0.115       -0.003

especificacao amplitude_pp        C2_menos_C1        C3_menos_C2       
                       min    max         min    max         min    max
formulacao                                                             
B                    0.077  0.120       0.074  0.108      -0.007  0.013
C                    0.075  0.115       0.068  0.115      -0.008  0.012


## 7. Recorte de suporte `[0,05; 0,95]`

Sensibilidade, não resultado principal: o recorte muda a população-alvo. N excluído e percentual referem-se ao escore da própria especificação (C1/C2 compartilham o logístico; C3 usa o HGB). `fora_001_099_n` é só diagnóstico.

In [ ]:
trim = pd.read_csv(pasta / 'trimming.csv')
display(trim.loc[trim.formulacao.isin(['B', 'C']) & trim.especificacao.isin(['C1', 'C3']),
                 ['formulacao', 'ano', 'especificacao', 'n_excluido', 'pct_excluido', 'estimativa_pp_sem_trimming',
                  'estimativa_pp_005_095', 'delta_trimming_pp', 'fora_001_099_n', 'status_suporte_09d']].round(3).reset_index(drop=True))
print(trim.loc[trim.formulacao.isin(['B', 'C'])].groupby('formulacao')[['pct_excluido', 'delta_trimming_pp']].agg(['min', 'max']).round(3))

,formulacao,ano,especificacao,n_excluido,pct_excluido,estimativa_pp_sem_trimming,estimativa_pp_005_095,delta_trimming_pp,fora_001_099_n,status_suporte_09d
0,B,2014,C1,162814,6.224,-0.936,-0.931,0.005,0,APROVADO
1,B,2014,C3,173263,6.624,-0.853,-0.872,-0.020,0,APROVADO
2,B,2015,C1,168488,6.303,-1.003,-0.999,0.004,0,APROVADO
3,B,2015,C3,174195,6.516,-0.920,-0.915,0.004,0,APROVADO
4,B,2016,C1,191177,7.465,-1.090,-1.094,-0.005,0,APROVADO
5,B,2016,C3,194234,7.585,-0.990,-1.020,-0.030,0,APROVADO
6,B,2017,C1,213502,8.115,-1.126,-1.114,0.012,0,APROVADO
7,B,2017,C3,212854,8.091,-1.036,-1.049,-0.013,0,APROVADO
8,B,2018,C1,227066,8.497,-1.256,-1.238,0.018,0,APROVADO
9,B,2018,C3,220813,8.263,-1.169,-1.162,0.007,0,APROVADO


           pct_excluido         delta_trimming_pp       
                    min     max               min    max
formulacao                                              
B                 6.224  10.556            -0.067  0.054
C                 8.140  14.935            -0.096  0.050


## 8. Inferência municipal versus iid

A leitura principal é o SE agrupado por município de residência (IC com t(G−1)); o iid é a referência histórica.

In [ ]:
inf = pd.read_csv(pasta / 'inferencia.csv')
inf = inf.loc[inf.formulacao.isin(['B', 'C']) & inf.trimming.eq('sem_trimming')]
display(inf.pivot_table(index='ano', columns=['formulacao', 'especificacao'], values='razao_se_cluster_iid').round(3))
display(inf.loc[inf.especificacao.eq('C1'), ['formulacao', 'ano', 'n_clusters', 'estimativa_pp', 'se_iid_pp', 'se_cluster_pp',
                                             'ic95_cluster_inferior_pp', 'ic95_cluster_superior_pp']].round(4).reset_index(drop=True))

formulacao         B                    C              
especificacao     C1     C2     C3     C1     C2     C3
ano                                                    
2014           1.136  1.132  1.147  1.089  1.090  1.094
2015           1.106  1.109  1.103  1.094  1.099  1.090
2016           1.377  1.394  1.400  1.260  1.271  1.279
2017           1.362  1.376  1.386  1.293  1.299  1.322
2018           1.448  1.453  1.474  1.368  1.367  1.379
2019           1.184  1.177  1.168  1.194  1.195  1.178
2020           1.313  1.312  1.349  1.277  1.283  1.299
2021           1.213  1.195  1.193  1.155  1.146  1.141
2022           1.198  1.189  1.196  1.105  1.099  1.093
2023           1.126  1.120  1.118  1.093  1.093  1.095
2024           1.105  1.108  1.122  1.037  1.041  1.060

,formulacao,ano,n_clusters,estimativa_pp,se_iid_pp,se_cluster_pp,ic95_cluster_inferior_pp,ic95_cluster_superior_pp
0,B,2014,5583,-0.9364,0.0491,0.0558,-1.0457,-0.8270
1,B,2015,5587,-1.0031,0.0496,0.0549,-1.1107,-0.8955
2,B,2016,5586,-1.0898,0.0520,0.0717,-1.2302,-0.9493
3,B,2017,5581,-1.1257,0.0532,0.0724,-1.2677,-0.9838
4,B,2018,5582,-1.2561,0.0539,0.0780,-1.4090,-1.1033
5,B,2019,5580,-1.2103,0.0554,0.0655,-1.3387,-1.0818
6,B,2020,5583,-1.0660,0.0559,0.0734,-1.2099,-0.9220
7,B,2021,5586,-1.2518,0.0596,0.0723,-1.3936,-1.1100
8,B,2022,5586,-1.5239,0.0628,0.0752,-1.6713,-1.3766
9,B,2023,5583,-1.4291,0.0638,0.0719,-1.5700,-1.2881


## 9. Diagnóstico de influência (descritivo)

Por ano e formulação, sem recorte: fração de IF² no 1% mais extremo (`top1_if2`), maior fração da variância municipal em um único município (U_g²) e razão SE municipal/iid. São descritores, sem gate automático; o histórico de 2024 (A) serve de referência.

In [ ]:
inf_ = pd.read_csv(pasta / 'influencia.csv')
display(inf_.loc[inf_.especificacao.eq('C1'), ['formulacao', 'ano', 'top1_if2', 'max_fracao_u2_municipio', 'municipio_maior_u2',
                                               'razao_se_cluster_iid', 'max_contribuicao_individual_pp', 'ess_t0_sobre_n']].round(4).reset_index(drop=True))
print(inf_.groupby(['formulacao', 'especificacao'])[['top1_if2', 'max_fracao_u2_municipio', 'razao_se_cluster_iid']].agg(['min', 'max']).round(3))

,formulacao,ano,top1_if2,max_fracao_u2_municipio,municipio_maior_u2,razao_se_cluster_iid,max_contribuicao_individual_pp,ess_t0_sobre_n
0,A,2024,0.7878,0.0542,211130,1.1015,0.0015,0.7520
1,B,2014,0.7543,0.0503,410480,1.1361,0.0016,0.6764
2,B,2015,0.7615,0.0917,330455,1.1063,0.0014,0.6815
3,B,2016,0.7707,0.1707,330455,1.3770,0.0015,0.6708
4,B,2017,0.7868,0.2006,350950,1.3621,0.0015,0.6643
5,B,2018,0.7941,0.1707,355030,1.4476,0.0016,0.6683
6,B,2019,0.7964,0.0771,431490,1.1836,0.0016,0.6831
7,B,2020,0.8023,0.2192,330455,1.3127,0.0015,0.6911
8,B,2021,0.8098,0.2056,355030,1.2130,0.0016,0.7024
9,B,2022,0.7949,0.1064,355030,1.1975,0.0016,0.7196


                         top1_if2        max_fracao_u2_municipio        razao_se_cluster_iid       
                              min    max                     min    max                  min    max
formulacao especificacao                                                                           
A          C1               0.788  0.788                   0.054  0.054                1.102  1.102
           C2               0.788  0.788                   0.053  0.053                1.097  1.097
           C3               0.790  0.790                   0.055  0.055                1.108  1.108
B          C1               0.754  0.819                   0.046  0.219                1.105  1.448
           C2               0.754  0.819                   0.052  0.232                1.108  1.453
           C3               0.756  0.819                   0.054  0.251                1.103  1.474
C          C1               0.791  0.848                   0.033  0.208                1.037  1.368


## 10. Leitura temporal

- **Mesma forma em A, B e C.** As três séries C1 têm o mesmo desenho: magnitude crescente de 2014 (B −0,94; C −0,85 p.p.) a 2018–2019, recuo em 2020 (B −1,07; C −0,96), novo aumento em 2021 e máximo em 2022 (B −1,52; C −1,49), seguido de 2023–2024 próximos de −1,4 a −1,5 em B e de −1,33 em C. O padrão já aparecia na V1 (notebook 09).
- **Nível: B é mais negativo que A e C.** Ano a ano, B fica de 0,09 a 0,14 p.p. abaixo da V1 histórica (A) e de 0,04 a 0,15 p.p. abaixo de C. Restringir o comparador a 4–6 aumenta a magnitude; restringir a 4–5 a reduz parcialmente. Os estimandos são distintos (população e comparador mudam), de modo que essas diferenças não medem viés diretamente.
- **Incerteza.** Todos os ICs95% municipais de B e C, nos 11 anos e nas três especificações, ficam abaixo de zero. A sobreposição ou separação de ICs entre anos não é um teste de mudança temporal.
- **Anos sinalizados.** B/2024 e C/2017–2024 estão sinalizados por caudas do escore no 09d. Neles o contraste segue a série vizinha, e o recorte `[0,05; 0,95]` muda pouco a estimativa (ver seção 7).
- **Sem atribuição.** O recuo de 2020 e o pico de 2022 são heterogeneidade temporal aparente; nenhum teste de quebra foi feito, e nada aqui os atribui a COVID-19, políticas ou mudanças de registro.

## 11. Limitações

- **Identificação.** São contrastes padronizados sob troca condicional nas sete X; confundimento residual (renda, acesso, intenção da gestação, morbidade prévia) é plausível.
- **Seleção.** A população é de nascidos vivos; a seleção por nascido vivo pode atuar como *collider*.
- **Mensuração.** `MESPRENAT` é retrospectivo; códigos 99/vazio/10 ficam fora de T, e o *missing* pode ser diferencial entre anos.
- **Tempo.** B e C reduzem, mas não eliminam, o desalinhamento temporal da exposição identificado no 09c; D (*landmark*) segue pendente.
- **Estimandos distintos.** A, B e C diferem em população e comparador; diferenças entre eles não medem viés de forma direta.
- **Inferência.** O IC municipal não cobre dependência entre anos nem toda a incerteza dos nuisances; ICs anuais sobrepostos não testam igualdade.
- **Mudanças anuais** não são atribuídas a COVID-19, políticas ou outros eventos: nenhum teste foi feito para isso.

## 12. Conclusão

1. **Execução.** O controle positivo A/2024 reconciliou com nuisances reajustados do zero (141 checagens, maior diferença absoluta 2,2e-16). B e C foram estimados nos 11 anos com C1/C2/C3 (414 ajustes, nenhum bloqueio, nenhuma falha de convergência).
2. **Modelo.** C1 é sistematicamente 0,07–0,12 p.p. mais negativo que C2/C3, e C2 ≈ C3 (diferença ≤ 0,015 p.p.); a amplitude anual entre as três vai de 0,08 a 0,12 p.p. É o mesmo padrão da V1, sem mudança de sinal nem de ordenação temporal.
3. **Recorte de suporte.** `[0,05; 0,95]` exclui 6–11% em B e 8–15% em C e altera C1 em no máximo 0,07 p.p. (B) e 0,10 p.p. (C, 2023). Não muda conclusões.
4. **Inferência e influência (descritivo).** A razão SE municipal/iid fica entre 1,04 e 1,45, e a concentração de IF² no 1% superior entre 0,75 e 0,85, semelhante à V1. Em alguns anos (2016–2018, 2020–2021) um único município (Rio de Janeiro ou São Paulo) responde por 17–22% da variância municipal, contra 5% em A/2024. Isso fica sinalizado para revisão posterior, sem novo gate.
5. **Leitura.** Sob as hipóteses de identificação, B (V2 principal) indica um contraste AIPW padronizado de cerca de −0,9 a −1,5 p.p. no risco de baixo peso para início do pré-natal em 1–3 em vez de 4–6, e C conta a mesma história temporal com magnitude ligeiramente menor. Não é prova causal: confundimento residual, seleção por nascido vivo, mensuração de `MESPRENAT` e o problema temporal residual permanecem.

In [ ]:
print('Controle positivo A/2024:', manifesto['controle_positivo_a2024'])
print('Baseline histórico:', manifesto['baseline_final'])
print('Contrato V2:', manifesto['contrato_v2_hash'])
print('Execuções:', len(manifesto['execucoes']), '| formulações x anos:',
      sorted({(e['formulacao'], e['ano']) for e in manifesto['execucoes']}) == sorted({('A', 2024)} | {(f, a) for f in 'BC' for a in range(2014, 2025)}))
assert manifesto['baseline_final'] == 'BASELINE_2024_PRESERVADO'

Controle positivo A/2024: CONTROLE_POSITIVO_A2024_APROVADO
Baseline histórico: BASELINE_2024_PRESERVADO
Contrato V2: 2174989c33b338f7215270050969efda1d3d8e09661aee97b1da547185927d0b
Execuções: 23 | formulações x anos: True
